In [ ]:
import pandas as pd

df = pd.read_csv("final_overall_result_location.csv")

df["old_value"] = df["old_value"].fillna("").astype(str).str.strip()
df["new_value"] = df["new_value"].fillna("").astype(str).str.strip()

def collect_values(series):
    return set(s.lower() for s in series if s and s.lower() != "nan")

grouped = (
    df.groupby("document_id")
    .agg(
        v1_values=("old_value", collect_values),
        v3_values=("new_value", collect_values),
        categories=("category", list)
    )
    .reset_index()
)

def calc_metrics(row):
    v1 = row["v1_values"]
    v3 = row["v3_values"]

    common  = v1 & v3
    added   = v3 - v1
    removed = v1 - v3

    cats = row["categories"]
    partial_count = cats.count("partial")

    return pd.Series({
        "v1_count": len(v1),
        "v3_count": len(v3),
        "common_count": len(common),
        "added_count": len(added),
        "removed_count": len(removed),
        "partial_count": partial_count,
        "common": sorted(common),
        "added": sorted(added),
        "removed": sorted(removed),
    })

result = pd.concat([grouped[["document_id"]], grouped.apply(calc_metrics, axis=1)], axis=1)

print(result.head(10))
result.to_csv("location_per_document.csv", index=False)

                document_id  v1_count  v3_count  common_count  added_count  \
0  6a7eb80195a7d25a93ea8b1b         1         0             0            0   
1  6a7eb80d95a7d25a93ea8b1c         2         2             2            0   
2  6a7eb81795a7d25a93ea8b1d         1         1             1            0   
3  6a7eb82095a7d25a93ea8b1e         1         1             1            0   
4  6a7eb83395a7d25a93ea8b20         1         1             0            1   
5  6a7eb83895a7d25a93ea8b21         1         0             0            0   
6  6a7eb84095a7d25a93ea8b22         1         1             1            0   
7  6a7eb84895a7d25a93ea8b23         1         1             1            0   
8  6a7eb84f95a7d25a93ea8b24         1         1             1            0   
9  6a7eb85795a7d25a93ea8b25         1         1             1            0   

   removed_count  partial_count                 common             added  \
0              1              0                     []           